# Banking Generative AI Chatbot

## Objective

Build a conversational AI banking chatbot using:

- LangChain
- FAISS
- Groq Llama 3.3
- Conversational Memory
- RAG Architecture

This notebook will:
- create conversational banking chatbot
- maintain chat history
- support contextual conversations
- generate intelligent responses
- prepare Streamlit deployment

---

## Technologies Used

- LangChain
- FAISS
- Groq LLM
- Conversational Memory
- Retrieval-Augmented Generation (RAG)

### Import Libraries

In [21]:
# Data Handling
import pandas as pd
import numpy as np

# Environment Variables
import os

# LangChain
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Groq LLM
from langchain_groq import ChatGroq

# Memory
from langchain.memory import ConversationBufferMemory

# Conversational Retrieval Chain
from langchain.chains import ConversationalRetrievalChain

# Prompt Template
from langchain.prompts import PromptTemplate

# Environment Variables
from dotenv import load_dotenv

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

### Load Environment Variables

In [22]:
load_dotenv()

# Groq API Key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Environment Variables Loaded Successfully")

Environment Variables Loaded Successfully


### Why Conversational AI?

Basic RAG:
```text
Question → Retrieval → Answer
```

Conversational RAG:
```text
Chat History
+ Current Question
+ Retrieval
→ Contextual AI Response
```

---

# Benefits

Conversational AI:
- remembers previous questions
- understands follow-up queries
- supports multi-turn conversations
- feels more human-like

---

# Example

User:
```text
What is a home loan?
```

Then:
```text
What happens if I miss EMI?
```

The chatbot understands:
```text
EMI related to home loan
```

using conversation history.

### Load Embedding Model

In [23]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

Embedding Model Loaded Successfully


### Load FAISS Vector Database

In [24]:
vectorstore = FAISS.load_local(
    "../vectorstore/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS Vector Database Loaded Successfully")

FAISS Vector Database Loaded Successfully


### Create Retriever

In [25]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever Created Successfully")

Retriever Created Successfully


### Load Groq LLM

In [26]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Loaded Successfully")

Groq LLM Loaded Successfully


#### Test LLM

In [ ]:
response = llm.invoke("Explain fixed deposit.")

print(response.content)

### Create Banking Assistant Prompt

#### Banking AI Prompt Engineering

Prompt engineering helps:
- improve answer quality
- reduce hallucinations
- enforce safe banking responses
- create professional AI behavior

The chatbot should:
- answer politely
- stay within banking context
- avoid false information
- use retrieved banking knowledge

#### Create Prompt Template

In [8]:
template = """

You are a professional banking AI assistant.

Use ONLY the provided banking context and conversation history.

Rules:
- Give concise and accurate answers
- Do not make up information
- If answer is unavailable, say:
  "I could not find relevant banking information."

Conversation History:
{chat_history}

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=template,
    input_variables=[
        "chat_history",
        "context",
        "question"
    ]
)

print("Prompt Template Created Successfully")

Prompt Template Created Successfully


#### Create Conversation Memory

In [9]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

print("Conversation Memory Created Successfully")

Conversation Memory Created Successfully


### Why Memory is Important?

Without memory:
```text
Every query is independent
```

With memory:
```text
Chatbot remembers previous interactions
```

This enables:
- contextual conversations
- follow-up questions
- intelligent banking discussions

---

# Example

User:
```text
Tell me about personal loans
```

Follow-up:
```text
What is the EMI?
```

The chatbot understands:
```text
EMI for personal loan
```

### Create Conversational RAG Chain

In [10]:
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs= {"prompt": PROMPT}
)

print("Conversational RAG Chain Created Successfully")

Conversational RAG Chain Created Successfully


### Understanding Conversational Pipeline

#### Conversational RAG Architecture

The chatbot now follows:

```text
User Question
↓
Conversation History
↓
Semantic Retrieval (FAISS)
↓
Relevant Banking Context
↓
Groq LLM
↓
Contextual AI Response
```

This architecture powers:
- enterprise AI assistants
- fintech chatbots
- AI customer support systems

### Test Chatbot Conversation

In [11]:
query = "What is a savings account?"

response = conversation_chain.invoke(
    {"question": query}
)

print(response["answer"])

A savings account lets you keep your money safe in a bank while earning a small interest, usually around 2–4% per year.


#### Ask Follow-Up Question

In [12]:
query = "What interest does it provide?"

response = conversation_chain.invoke(
    {"question": query}
)

print(response["answer"])

A savings account usually provides an interest of around 2–4% per year.


### Another Conversation Test

In [13]:
query = "Explain home loan"

response = conversation_chain.invoke(
    {"question": query}
)

print(response["answer"])

A home loan is money borrowed from a bank to buy or build a house. You repay it in monthly instalments over 10–30 years.


#### Follow-Up on EMI

In [14]:
query = "What happens if I miss EMI payment?"

response = conversation_chain.invoke(
    {"question": query}
)

print(response["answer"])

If a monthly installment payment for a loan is missed, the bank may send a legal notice warning to clear dues if several EMIs are missed, which can lead to recovery action or asset seizure.


### Display Source Documents

In [15]:
docs = response["source_documents"]

for i, doc in enumerate(docs):
    print("="*80)
    print(f"Source Document {i+1}")
    print("="*80)

    print(doc.page_content)
    print("\n")

Source Document 1
Question: How does a loan default notice work?
Answer: Typically, If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring it can lead to recovery action or asset seizure.


Source Document 2
Question: How does What is a loan default notice work? #12072
Answer: If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring it can lead to recovery action or asset seizure.


Source Document 3
Question: How does the bank handle What is a loan default notice? #40161
Answer: If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring it can lead to recovery action or asset seizure.


Source Document 4
Question: Can you explain a loan default notice?
Answer: In simple words, If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring it can lead to recovery action or asset seizure.


Source Document 5
Question: Please explain a loan default notice.

### View Chat History

In [16]:
memory.chat_memory.messages

[HumanMessage(content='What is a savings account?'),
 AIMessage(content='A savings account lets you keep your money safe in a bank while earning a small interest, usually around 2–4% per year.'),
 HumanMessage(content='What interest does it provide?'),
 AIMessage(content='A savings account usually provides an interest of around 2–4% per year.'),
 HumanMessage(content='Explain home loan'),
 AIMessage(content='A home loan is money borrowed from a bank to buy or build a house. You repay it in monthly instalments over 10–30 years.'),
 HumanMessage(content='What happens if I miss EMI payment?'),
 AIMessage(content='If a monthly installment payment for a loan is missed, the bank may send a legal notice warning to clear dues if several EMIs are missed, which can lead to recovery action or asset seizure.')]

### Clear Chat Memory

In [17]:
memory.clear()

print("Conversation Memory Cleared")

Conversation Memory Cleared


### Production Chatbot Function

In [18]:
def banking_chatbot(query):
    response = conversation_chain.invoke(
        {"question": query}
    )

    return {
        "question": query,
        "answer": response["answer"],
        "source_documents":response["source_documents"]
    }

### Final Chatbot Testing

In [19]:
query = "How to activate mobile banking?"

result = banking_chatbot(query)

print("Question:")
print(result["question"])

print("\n")

print("Answer:")
print(result["answer"])

Question:
How to activate mobile banking?


Answer:
Download your bank's official app, enter your account details, verify with an OTP on your registered mobile, and set your MPIN.


### Multi-Turn Conversation Demo

In [20]:
queries = [
    "What is a credit card?",
    "How is it different from debit card?",
    "What happens if payment is delayed?"
]

for q in queries:
    print("="*80)

    print("User:")
    print(q)

    response = banking_chatbot(q)

    print("\nAI Assistant:")
    print(response["answer"])

    print("\n")

User:
What is a credit card?

AI Assistant:
A credit card lets you borrow money from the bank to make purchases. You pay it back later, and if you miss the deadline, you're charged interest.


User:
How is it different from debit card?

AI Assistant:
The main difference is that a credit card lets you borrow money from the bank, while a debit card uses the money directly from your bank account. With a credit card, you pay back the borrowed amount later, potentially with interest, whereas with a debit card, the funds are deducted immediately from your account.


User:
What happens if payment is delayed?

AI Assistant:
If a credit card payment is delayed, you're charged interest.




# Key Insights

## 1. Conversational Banking AI Successfully Built

The project now supports:
- conversational AI
- chat history
- contextual understanding
- intelligent follow-up handling

---

## 2. Memory-Enabled Chatbot

Using:
```python
ConversationBufferMemory
```

the chatbot can:
- remember previous questions
- understand context
- support multi-turn conversations

This creates a more human-like AI assistant.

---

## 3. Enterprise-Level GenAI Architecture

The system now combines:
- semantic retrieval
- FAISS vector database
- Groq LLM
- conversational memory
- RAG pipelines

This resembles real-world:
- fintech AI systems
- enterprise copilots
- banking support assistants

---

## 4. Hallucination Reduction

The chatbot generates grounded responses because:
- retrieval provides real banking context
- prompts restrict hallucinations
- LLM answers are context-aware

---

## 5. Lightweight Yet Powerful System

The architecture remains laptop-friendly because:
- embeddings are lightweight
- FAISS is efficient
- Groq performs remote inference

No local GPU is required.